In [3]:
import os
import cv2
import subprocess
import numpy as np
from ultralytics import YOLO
import re

# ---------------------------
# Settings and configuration
# ---------------------------
VODS_FOLDER = "VODS"              # Folder containing the videos
KILLFEED_FOLDER = "killfeed/images"      # Folder to store cropped killfeed frames
SAMPLED_LOG = "killfeed/sampled_videos_killfeed.txt" # Log file to track already processed videos
NUM_SAMPLES = 30                  # Number of frames to sample per video

# Crop coordinates for each area (x, y, width, height)
crop_areas = {
    "minimap": (20, 30, 300, 275),
    "score": (360, 0, 500, 150),
    "killfeed": (850, 50, 430, 350)
}

def format_video_name(video_name, frame_number):
    """
    Given a video name (string) and a frame number,
    returns a filename formatted as: firstword_thirdword_killfeed_xxx.png
    """
    parts = re.split(r" - | ", video_name)
    if len(parts) < 3:
        # Fallback if there are not enough parts
        return f"{video_name}_killfeed_{frame_number:03d}.png"
    
    first_word = parts[0]
    third_word = parts[2]
    last_two_words = "_".join(parts[-2:]).lower()  # Last two words (e.g., "map_01")
    return f"{first_word}_{third_word}_{last_two_words}_killfeed_{frame_number:03d}.png"

# Create killfeed folder if it doesn't exist
os.makedirs(KILLFEED_FOLDER, exist_ok=True)

# Load the trained YOLO classification model
classify_model = YOLO("runs/classify/train/weights/best.pt")

# ---------------------------
# Load processed videos log
# ---------------------------
if os.path.exists(SAMPLED_LOG):
    with open(SAMPLED_LOG, "r") as f:
        processed_videos = set(f.read().splitlines())
else:
    processed_videos = set()

# ---------------------------
# Process each video in VODS
# ---------------------------
video_files = [f for f in os.listdir(VODS_FOLDER) if f.endswith((".mp4", ".webm"))]

for video in video_files:
    if video in processed_videos:
        print(f"Skipping {video} (already processed).")
        continue

    video_path = os.path.join(VODS_FOLDER, video)
    
    # Use ffprobe to get video duration
    cmd = f'ffprobe -i "{video_path}" -show_entries format=duration -v quiet -of csv="p=0"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    try:
        video_length = float(result.stdout.strip())
    except ValueError:
        print(f"Could not determine duration for {video}. Skipping.")
        continue

    # Generate NUM_SAMPLES timestamps using a normal distribution centered at the middle
    mu = video_length / 2.0
    sigma = video_length / 4.0  # Adjust sigma if needed
    timestamps = np.random.normal(mu, sigma, NUM_SAMPLES)
    timestamps = np.clip(timestamps, 0, video_length)  # Ensure timestamps are within video bounds
    timestamps = np.sort(timestamps)  # Optional: sort for chronological order

    # Open video with OpenCV
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Failed to open {video_path}. Skipping.")
        continue

    video_name = os.path.splitext(video)[0]
    sample_index = 100  # Counter for naming output frames

    for idx, timestamp in enumerate(timestamps):
        # Set video to the desired timestamp (in milliseconds)
        cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
        ret, frame = cap.read()
        if not ret:
            print(f"Could not read frame at {timestamp:.2f}s from {video}.")
            continue

        # Convert the frame to RGB (if your model expects RGB)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = classify_model(rgb_frame)
        
        # Retrieve the classification prediction
        prediction = results[0].probs.top1.item() if hasattr(results[0].probs.top1, "item") else results[0].probs.top1
        
        # If the classification prediction equals 1, crop to killfeed area and save the image
        if prediction == 1:
            # Get killfeed crop coordinates
            x, y, w, h = crop_areas["killfeed"]
            if x + w > frame.shape[1] or y + h > frame.shape[0]:
                print(f"Killfeed crop out of bounds for frame from {video} at {timestamp:.2f}s.")
                continue

            # Within your processing loop:
            cropped_frame = frame[y:y+h, x:x+w]
            custom_filename = format_video_name(video_name, sample_index)
            save_path = os.path.join(KILLFEED_FOLDER, custom_filename)
            cv2.imwrite(save_path, cropped_frame)
            print(f"Saved cropped killfeed frame at {timestamp:.2f}s from {video} as {custom_filename} (prediction: {prediction}).")
            sample_index += 1

    cap.release()

    # Log the processed video to avoid reprocessing later
    with open(SAMPLED_LOG, "a") as f:
        f.write(video + "\n")
    print(f"Finished processing {video}.\n")

# ---------------------------
# Launch LabelImg for annotation
# ---------------------------
print("Launching LabelImg on the killfeed folder...")
subprocess.run(["labelImg", KILLFEED_FOLDER])


Skipping cropped_useful.mp4 (already processed).
Skipping G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 04.webm (already processed).
Skipping G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 05.mp4 (already processed).
Skipping SEN vs. TL - VALORANT Masters Bangkok - Swiss Stage - Map 03.webm (already processed).
Skipping TE vs. T1 - VALORANT Masters Bangkok - Swiss Stage - Map 02.webm (already processed).
Skipping test.webm (already processed).
Skipping VIT vs. DRX - VALORANT Masters Bangkok - Swiss Stage - Map 01.webm (already processed).
Launching LabelImg on the killfeed folder...


CompletedProcess(args=['labelImg', 'killfeed/images'], returncode=0)